In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 日本語フォント設定
# plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")
plt.style.use("default")

In [ ]:
# 日本語フォント設定
matplotlib.rc("font", family="IPAexGothic")

In [ ]:
# 現在の最大表示列数の出力
pd.get_option("display.max_columns")

In [ ]:
# 最大表示列数の指定
pd.set_option("display.max_columns", 150)

In [ ]:
# 現在の最大表示行数の出力
pd.get_option("display.max_rows")

In [ ]:
# 最大表示行数の指定
pd.set_option("display.max_rows", 40)

In [ ]:
# -------------------------------------------------------------------------
# 選手登番・艇番ごとの成績読み込み
# -------------------------------------------------------------------------

In [ ]:
racers_stats_lane_df = pd.read_csv("data/racers_stats_lane.csv")
racers_stats_lane_df.head()

In [ ]:
# -------------------------------------------------------------------------
# 予測用のpred.csvデータを作成する
# -------------------------------------------------------------------------

In [ ]:
programs_df = pd.read_csv("data/programs.csv")
programs_df.head()

In [ ]:
# -------------------------------------------------------------------------
# 必要な列データの作成
# -------------------------------------------------------------------------

In [ ]:
# "年" "月" "日" 列から日付列を作成
# 年,月,日を結合して日付型に変換
programs_df.insert(
    0,
    "開催日",
    pd.to_datetime(
        programs_df["年"].astype(str)
        + "-"
        + programs_df["月"].astype(str)
        + "-"
        + programs_df["日"].astype(str)
    ),
)

In [ ]:
# "年" "月" "日" "レース場番号" "レース番号"からレースID列を作成して先頭に挿入
programs_df.insert(
    1,
    "レースID",
    programs_df["年"].astype(str)
    + programs_df["月"].astype(str).str.zfill(2)
    + programs_df["日"].astype(str).str.zfill(2)
    + programs_df["レース場番号"].astype(str).str.zfill(2)
    + programs_df["レース番号"].astype(str).str.zfill(2),
)

In [ ]:
# 枠番を艇番に変更
programs_df = programs_df.rename(columns={"枠番": "艇番"})

In [ ]:
# 不要列を削除
programs_df = programs_df.drop(columns=["年", "月", "日"])

In [ ]:
# -------------------------------------------------------------------------
# 指定日とレース場番号を指定して必要な列データの作成
# -------------------------------------------------------------------------

In [ ]:
pred_date = "2025-11-08"
pred_racecourse_number = 22

In [ ]:
programs_df = programs_df[
    (programs_df["開催日"] == pd.to_datetime(pred_date))
    & (programs_df["レース場番号"] == pred_racecourse_number)
]
programs_df.head()

In [ ]:
# 必要な列データの抽出して、コース別選手情報とマージする
pred_df = programs_df[["レースID", "選手登番", "艇番"]].merge(
    racers_stats_lane_df[["勝率", "複勝率", "平均レースタイム秒", "選手登番", "艇番"]],
    on=["選手登番", "艇番"],
    how="left",
)

In [ ]:
# レースID内の艇番ごとの勝率差を計算して新しい特徴量を作成
pred_df["勝率差"] = pred_df.groupby("レースID")["勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの複勝率差を計算して新しい特徴量を作成
pred_df["複勝率差"] = pred_df.groupby("レースID")["複勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの平均レースタイム秒差を計算して新しい特徴量を作成
pred_df["平均レースタイム秒差"] = pred_df.groupby("レースID")[
    "平均レースタイム秒"
].transform(lambda x: x - x.mean())

In [ ]:
# 不要カラムを削除
pred_df = pred_df.drop(columns=["勝率", "複勝率", "平均レースタイム秒"])

In [ ]:
pred_df.head(40)

In [ ]:
# -------------------------------------------------------------------------
# pred.csvデータの保存
# -------------------------------------------------------------------------
pred_df.to_csv("data/pred_data.csv", index=False)